# HVG-count sweep — build the 1k / 2k / 3k gene-set variants

Builds one independent variant per HVG count, so [`verify_variants`](verify_variants.ipynb) §9 can ask
whether either representation has a preferred gene-set size under cross-validation.

**This is not a pipeline stage, and that is the point.** It was §B of the old
`1_preprocessing.ipynb` (now [archived](../../archive/1_preprocessing.ipynb)) and moved here on
12.08.2026 (Selin): a stage everyone runs should not carry an off-by-default branch that exists for
one analysis. The numbered chain builds the one variant the project runs on; this builds the extra
ones the sweep compares against.

**It runs the same code the pipeline does** — `scripts/preprocessing/pipeline.py`, step by step — so
a sweep variant is built exactly as `hvg5000` was and the comparison is not confounded by how the
inputs were made.

⚠️ **Expensive: hours and gigabytes.** Each variant re-embeds every cell with scGPT. It is gated
behind `RUN_HVG_SWEEP` and skips any variant whose targets file already exists.

**`hvg5000` and `all_genes` are not built here** — the first is the working variant from
[`1_data`](../../1_data.ipynb) + [`3_representations`](../../3_representations.ipynb), the second was
built alongside it.

**The split is shared, not redrawn.** `pipeline.splits` reads the frozen `splits/split_ctrp.csv`, so
every variant places the same cell lines in train, val and test. That is what makes the sweep a
comparison of gene sets rather than of partitions.

> ⛔ **Nothing here has been re-run**, and this notebook in particular must not be: the 03.08.2026
> freeze in [TODO](../../../docs/TODO.md) holds until the review finishes, and rebuilding embeddings
> is exactly what it forbids.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.layout import PipelinePaths
from scripts.preprocessing import add_pca, pipeline

# ⚠️ True for the 13.08.2026 re-embed (Selin). This REVERSES R1, which chose hvg5000 + all_genes
# over "all five" on cost grounds. The reason it is reversed: verify_variants §9 plots performance
# against gene-set size, and with only two variants rebuilt its five points differed in gene count
# AND in code version, so no trend could be read off the curve at all. Rebuilding the three makes
# the sweep like-for-like. **Set this back to False once the sweep exists.**
RUN_HVG_SWEEP = True
SCGPT_PYTHON = '/Users/selin/PycharmProjects/scGPT/.venv/bin/python'
SWEEP_VARIANTS = ['hvg1000', 'hvg2000', 'hvg3000']
PCA_N_COMPS = add_pca.DEFAULT_N_COMPS      # 512, matching the scGPT width

for v in SWEEP_VARIANTS:
    p = PipelinePaths.build(None, v)
    print(f'  {v:9s} {"present" if p.targets_h5ad.exists() else "MISSING"}')

## Build

Each variant runs the full chain: `fetch` (cached and idempotent, so it costs nothing after the
first) → `convert` → `scgpt` → `targets` → `splits` → `pca`.

A variant whose targets file already exists is skipped entirely rather than rebuilt. If one is
**partially** built — say `convert` ran but `scgpt` did not — the guard on the completed step raises
`FileExistsError` rather than overwriting it, and the fix is to decide deliberately: pass
`overwrite=True` for that step, or delete the partial variant directory. It is not silently resumed,
because a half-built variant that silently completes is a variant nobody can date.

In [ ]:
if not RUN_HVG_SWEEP:
    print('RUN_HVG_SWEEP is False -- skipping the heavy scGPT sweep build.')
else:
    for variant in SWEEP_VARIANTS:
        paths = PipelinePaths.build(None, variant)
        # ⚠️ The "targets present -> skip" guard is DELIBERATELY GONE (13.08.2026, Selin).
        # It read:
        #     if paths.targets_h5ad.exists():
        #         print(f'{variant}: targets present, skipping.'); continue
        # which made this cell a no-op for exactly the case it is now being run for: all three
        # variants already had targets, built 01:53 on 13.08.2026 by the pre-correction code. With
        # the guard in place, flipping RUN_HVG_SWEEP and passing overwrite=True would still have
        # skipped every variant and reported success -- the same shape of silent no-op this review
        # has been finding all day. RUN_HVG_SWEEP is now the only switch: True means rebuild.
        print('\n' + '=' * 70 + f'\n{variant}\n' + '=' * 70)
        pipeline.fetch(paths)
        # overwrite=True on the two guarded steps, because replacing these artifacts IS the point.
        # convert() attaches total_counts / pct_counts_mt itself, so the rebuilt variants carry the
        # UMI covariates the 01:53 ones lack -- which is half of what made the sweep unreadable.
        pipeline.convert(paths, overwrite=True)
        pipeline.scgpt(paths, SCGPT_PYTHON, overwrite=True)
        pipeline.targets(paths)
        pipeline.splits(paths)
        pipeline.pca(paths, n_comps=PCA_N_COMPS)
        print(f'{variant}: built -> {paths.targets_h5ad}')